# Predicting votes of individual members of parliament

Things  to do:
* sklearn nlp transformers
* extract `party_is_in_government` feature for members from legislature id
* extract `bill_proposed_by_government` feature from poll description
* depending on performance try encoding of poll description / poll title with language models

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn import (
    compose,
    dummy,
    ensemble,
    metrics,
    pipeline,
    preprocessing,
    set_config,
)

import bundestag.modelling as modelling

In [ ]:
set_config(transform_output="pandas")

In [ ]:
aw_path = Path("../data/preprocessed/abgeordnetenwatch")

In [ ]:
all_votes = pd.concat(
    [
        pd.read_parquet(aw_path / "df_all_votes_111.parquet"),
        pd.read_parquet(aw_path / "df_all_votes_132.parquet"),
    ],
    axis=0,
    ignore_index=True,
)
display(all_votes.head(), all_votes.tail())

In [ ]:
all_votes["politician name"] = all_votes["mandate"].str.extract(
    r"(.*)\s\(Bundestag", expand=True
)[0]

In [ ]:
polls = pd.concat(
    [
        pd.read_parquet(aw_path / "df_polls_111.parquet"),
        pd.read_parquet(aw_path / "df_polls_132.parquet"),
    ],
    ignore_index=True,
    axis=0,
)
display(polls.head(), polls.tail())

In [ ]:
for _, row in polls[["poll_title", "poll_description"]].iterrows():
    print(row["poll_title"])
    # print(row["poll_description"])
    print("\n".join(row["poll_description"].split("\n")[:-1]))
    print()

In [ ]:
polls["poll_date"] = pd.to_datetime(polls["poll_date"], format="%Y-%m-%d")
polls["poll_date"].head()

In [ ]:
polls["poll_description_without_results"] = polls["poll_description"].apply(
    lambda x: "\n".join(x.split("\n")[:-1])
)
polls["poll_description_without_results"].head()

In [ ]:
mandates = pd.concat(
    [
        pd.read_parquet(aw_path / "df_mandates_111.parquet"),
        pd.read_parquet(aw_path / "df_mandates_132.parquet"),
    ],
    ignore_index=True,
    axis=0,
)
display(mandates.head(2), mandates.tail(2))

In [ ]:
all_info = all_votes.merge(
    polls[
        [
            "poll_id",
            "poll_date",
            "poll_title",
            "poll_description_without_results",
            "legislature_period",
        ]
    ],
    on="poll_id",
    how="left",
)
all_info = all_info.merge(
    mandates[["mandate_id", "party"]], on="mandate_id", how="left"
)

all_info.head()

In [ ]:
target = "vote"

In [ ]:
y = all_info[target].copy()
y.head()

In [ ]:
x_cols = [
    "mandate_id",
    "poll_date",
    "poll_title",
    "poll_description_without_results",
    "legislature_period",
    "party",
    "poll_id",
]
X = all_info[x_cols].copy()
X.head()

## Time holdout split

In [ ]:
t_split = "2022-01-01"
t_split

In [ ]:
mask_train = X["poll_date"] < t_split
mask_test = ~mask_train

X0 = X.loc[mask_train, :].copy()
X1 = X.loc[mask_test, :].copy()
y0 = y.loc[mask_train].copy()
y1 = y.loc[mask_test].copy()

## Create a baseline model

In [ ]:
dummy_model = dummy.DummyClassifier(strategy="prior")
dummy_model.fit(X0, y0)

In [ ]:
performance_dummy = metrics.classification_report(
    y1, dummy_model.predict(X1), output_dict=True
)

print(json.dumps(performance_dummy, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1162
  },
  "no": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 16203
  },
  "no_show": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4978
  },
  "yes": {
    "precision": 0.5255759634780762,
    "recall": 1.0,
    "f1-score": 0.6890197224657953,
    "support": 24752
  },
  "accuracy": 0.5255759634780762,
  "macro avg": {
    "precision": 0.13139399086951906,
    "recall": 0.25,
    "f1-score": 0.17225493061644884,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.27623009338590815,
    "recall": 0.5255759634780762,
    "f1-score": 0.3621322044903571,
    "support": 47095
  }
}
```

## Hand-crafted features
> `party_is_in_government` and `bill_proposed_by_government` and `party`

### Tree-based

In [ ]:
X0.head()

In [ ]:
X0["legislature_period"].nunique()

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "mandate_id",
                    "poll_title",
                    "poll_description",
                    "legislature_period",
                    "poll_id",
                ]
            ),
        ),
        (
            "encode",
            compose.make_column_transformer(
                (
                    preprocessing.OneHotEncoder(sparse_output=False),
                    [
                        "party",
                    ],
                ),
                remainder="passthrough",
            ),
        ),
    ]
)

trafo.fit_transform(X0).head()

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        ("estimator", ensemble.RandomForestClassifier()),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_handcrafted = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_handcrafted, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1162
  },
  "no": {
    "precision": 0.4192466131566654,
    "recall": 0.6092698882922916,
    "f1-score": 0.4967044025157233,
    "support": 16203
  },
  "no_show": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4978
  },
  "yes": {
    "precision": 0.596526244267029,
    "recall": 0.5675096961861668,
    "f1-score": 0.5816563146997928,
    "support": 24752
  },
  "accuracy": 0.5078883108610256,
  "macro avg": {
    "precision": 0.2539432143559236,
    "recall": 0.2941948961196146,
    "f1-score": 0.269590179303879,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.4577613434775444,
    "recall": 0.5078883108610256,
    "f1-score": 0.4765953611935776,
    "support": 47095
  }
}
```

### FastAI-based

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "poll_title",
                    "poll_description",
                ]
            ),
        ),
    ]
)

trafo.fit_transform(X0).head()

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        (
            "estimator",
            modelling.FastAIClassifier(
                cat_names_endings=[
                    "mandate_id",
                    "party_is_in_government",
                    "party",
                    "bill_proposed_by_government",
                    "legislature_period",
                ]
            ),
        ),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_handcrafted = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_handcrafted, indent=2))

```json
{
  "abstain": {
    "precision": 0.08333333333333333,
    "recall": 0.15834767641996558,
    "f1-score": 0.10919881305637982,
    "support": 1162
  },
  "no": {
    "precision": 0.3260384956494643,
    "recall": 0.8394741714497316,
    "f1-score": 0.4696661026898242,
    "support": 16203
  },
  "no_show": {
    "precision": 0.1884469696969697,
    "recall": 0.11992768179991965,
    "f1-score": 0.14657500613798183,
    "support": 4978
  },
  "yes": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 24752
  },
  "accuracy": 0.3054039706975263,
  "macro avg": {
    "precision": 0.14945469966994182,
    "recall": 0.27943738241740423,
    "f1-score": 0.18135998047104646,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.1341485103194207,
    "recall": 0.3054039706975263,
    "f1-score": 0.1797757567302178,
    "support": 47095
  }
}
```

### Tree-based with TFIDF

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(
                description_col="poll_description_without_results"
            ),
        ),
        (
            "tfidf",
            modelling.DenseTfidfVectorizer(
                text_col="poll_description_without_results",
                poll_id_col="poll_id",
                no_below=10,
                no_above=0.5,
                keep_n=500,
            ),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "mandate_id",
                    "poll_description_without_results",
                    "legislature_period",
                    "poll_id",
                    "poll_title",
                ]
            ),
        ),
        (
            "encode",
            compose.make_column_transformer(
                (
                    preprocessing.OneHotEncoder(sparse_output=False),
                    [
                        "party",
                    ],
                ),
                remainder="passthrough",
            ),
        ),
    ]
)

trafo.fit_transform(X0).head()

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        ("estimator", ensemble.RandomForestClassifier()),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_tfidf_tree = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_tfidf_tree, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1162
  },
  "no": {
    "precision": 0.4997496602532008,
    "recall": 0.4312164413997408,
    "f1-score": 0.4629605088788762,
    "support": 16203
  },
  "no_show": {
    "precision": 0.10076992753623189,
    "recall": 0.08939333065488148,
    "f1-score": 0.09474132424952098,
    "support": 4978
  },
  "yes": {
    "precision": 0.5899017353125653,
    "recall": 0.6839447317388494,
    "f1-score": 0.6334518241347054,
    "support": 24752
  },
  "accuracy": 0.517273595923134,
  "macro avg": {
    "precision": 0.2976053307754995,
    "recall": 0.30113862594836793,
    "f1-score": 0.29778841431577563,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.4926281812679603,
    "recall": 0.517273595923134,
    "f1-score": 0.5022225499195407,
    "support": 47095
  }
}
```

### FastAI-based with TFIDF

In [ ]:
# TODO: CONTINUE HERE - fastai without overfitting? seems to immediately overfit

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(
                description_col="poll_description_without_results"
            ),
        ),
        (
            "tfidf",
            modelling.DenseTfidfVectorizer(
                text_col="poll_description_without_results",
                poll_id_col="poll_id",
                no_below=10,
                no_above=0.5,
                keep_n=500,
            ),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "poll_description_without_results",
                    "poll_title",
                ],
                column_beginnings=[],
            ),
        ),  # "tfidf_poll_title"
    ]
)

X0_trafo = trafo.fit_transform(X0)
X0_trafo.head(2)

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        (
            "estimator",
            modelling.FastAIClassifier(
                cat_names_endings=[
                    "mandate_id",
                    "party_is_in_government",
                    "party",
                    "bill_proposed_by_government",
                    "legislature_period",
                ],
                cont_names_beginnings=["tfidf_poll_description_"],
                batch_size=2048,
                n_epochs=5,
            ),
        ),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_tfidf_fastai = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_tfidf_fastai, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1162
  },
  "no": {
    "precision": 0.4192466131566654,
    "recall": 0.6092698882922916,
    "f1-score": 0.4967044025157233,
    "support": 16203
  },
  "no_show": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4978
  },
  "yes": {
    "precision": 0.596526244267029,
    "recall": 0.5675096961861668,
    "f1-score": 0.5816563146997928,
    "support": 24752
  },
  "accuracy": 0.5078883108610256,
  "macro avg": {
    "precision": 0.2539432143559236,
    "recall": 0.2941948961196146,
    "f1-score": 0.269590179303879,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.4577613434775444,
    "recall": 0.5078883108610256,
    "f1-score": 0.4765953611935776,
    "support": 47095
  }
}
```